# Welcome to Week 5 - Agent Frameworks

## Day 3: Agno

Same week, same five steps, a new framework. The whole idea this week is that once you know one agent framework, you basically know them all, so every day we build the same agent the same five steps and watch the idioms rhyme:

1. **Create the agent** - give it a model and a system prompt.
2. **Run it** - send a message, get a reply.
3. **Add tools** - plain typed functions the agent can call.
4. **Add MCP** - connect a tool server someone else wrote, wired the same way every time.
5. **Put it in a loop with a goal** - hand it an objective and let it work, step after step, until the job is done.

Steps 1 and 2 are still just an LLM call. Tools and MCP give the agent things it can do. Step 5 is where it becomes an agent: the framework runs the loop on its own, choosing a tool, reading the result, choosing again, until the goal is met.

The running project is the same SQLite todo board from Day 1. A worker picks one goal off the board, plans its own steps, does the work with its agent loop, and ticks each step off. The board code (`board.py`) is the same file, byte for byte; only the framework around it changes.

Today is **Agno**, from the team formerly behind Phidata, who call it "the programming language for agentic software." Two things make it stand out. It is strikingly fast and lean to instantiate, and the same agent you write here drops unchanged into AgentOS, its own FastAPI runtime with sessions, tracing and a control-plane UI. We use the plain SDK and clock the instantiation speed at the end. Notice how little changes from the other frameworks; that is the point.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The Agno docs</h2>
            <span style="color:#00bfff;">The docs live at <a href="https://docs.agno.com/introduction">docs.agno.com</a>. Agno's v2 (the AgentOS rewrite) renamed a good deal from v1, so we pin the version (2.6.14 here) and prefer the current pages over older Phidata-era posts. A model is an <code>OpenAIChat(id=...)</code> and an agent is an <code>Agent(model=..., tools=[...])</code>. One thing to know: connecting MCP makes the agent run async, so we <code>await</code> it with <code>arun</code>.</span>
        </td>
    </tr>
</table>

## Setup

Two things this day needs, both already in place from earlier weeks:

- **Node**, for `npx` (the filesystem MCP server runs over it). Check with `node --version`.
- **`OPENAI_API_KEY`** in the repo-root `.env`. Today's frameworks use OpenAI's `gpt-5.4-mini`, and that key has been in your `.env` since Week 1.

Agno is part of the repo's environment, so a normal `uv sync` from the repo root installs everything. Open this notebook in Cursor and pick the repo's default **Python 3.12.12** kernel, the same one you use every week, then run the cells top to bottom.

Warm the filesystem MCP server once now so the first run is fast, then stop it with Ctrl-C as soon as it prints that it is running:

```bash
npx -y @modelcontextprotocol/server-filesystem .
```

In [1]:
import functools
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.mcp import MCPTools
from mcp import StdioServerParameters

load_dotenv(override=True)

True

## Step 1: Create the agent

In Agno an agent is an `Agent`: a `model` plus `instructions`, its system prompt. We build the model once as an `OpenAIChat` pointed at `gpt-5.4-mini`, reading `OPENAI_API_KEY` from the environment, and reuse it. Agno has a synchronous path too, but because connecting MCP later makes the agent async, we `await` it throughout with `arun` so the whole notebook reads the same way.

In [10]:
# 사용할 모델 이름을 상수로 지정한다.
MODEL = "gpt-5-mini"

# 지정한 id의 OpenAI 채팅 모델 객체를 생성한다.
model = OpenAIChat(id=MODEL)

# 모델과 시스템 지시문을 묶어 에이전트를 생성한다.
agent = Agent(
    model=model,
    # 에이전트의 역할과 응답 방식을 정의하는 시스템 지시문 (한글)
    instructions="당신은 간결하고 친절한 어시스턴트입니다. 짧은 한 문장으로 답하십시오.",
)

## Step 2: Run it

Send a message, await the reply, and print the result's `.content`. With no tools yet there is nothing to loop over, so the agent just answers. This is still only an LLM call.

In [11]:
# 에이전트에 입력 메시지를 전달하고 비동기로 실행한 뒤 결과를 받는다.
result = await agent.arun(input="스페인어로 인사말을 해줘.")

# 실행 결과에서 모델이 생성한 텍스트 내용을 출력한다.
print(result.content)

¡Hola! ¿Cómo estás?


## Our project this week: a SQLite todo board

The worker coordinates through the same tiny SQLite board as Day 1, the same `board.py` file: one file, one table, no server to run. A worker is handed one **goal**; to reach it, it writes its own **step** todos under that goal, ticks each one off as it goes, and closes the goal at the end. Under the hood the board is just a list of dicts (a goal has `parent_id` of None, a step points at its goal):

In [12]:
# 보드 상태를 읽고 쓰기 위한 모듈을 가져온다.
import board

# 보드의 기존 내용을 모두 비우고 초기 상태로 되돌린다.
board.reset_board()

# 보드에 목표(goal)를 하나 추가한다. 목표는 parent_id가 None인 최상위 항목이다.
board.add_goal("notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 작성하라.")

# 현재 보드에 등록된 모든 todo 항목을 조회한다.
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 작성하라.',
  'status': 'pending',
  'result': ''}]

`show_board()` prints that same data nicely, in the rich style from Week 1: each goal with its steps indented beneath, done tasks struck through in green and the one in progress in yellow. There are no steps yet; the worker writes its own when it plans.

In [13]:
board.show_board()

Goal #1: notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 작성하라.

## Step 3: Add tools

A tool in Agno is a plain typed Python function with a docstring. Agno reads the type hints and the docstring and builds the schema for you, so there is nothing else to declare; an optional `@tool` decorator exists only for extras like caching. You hand the functions to the agent in its `tools=[...]` list.

We write three small board tools: `show_todos` to read the board, `plan_steps` to break a goal into steps, and `complete_task` to mark a todo done. Here we give a quick agent just two of them and ask what is on the board; watch it decide, on its own, to call `show_todos` before it answers. That decide, call, read, answer is the agent loop starting to turn. All three tools come together in step 5.

In [14]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [15]:
# 모델, 시스템 지시문, 사용할 도구 목록을 묶어 보드 관리용 에이전트를 생성한다.
board_agent = Agent(
    model=model,
    # 에이전트의 역할을 정의하는 시스템 지시문 (한글)
    instructions="당신은 공유 todo 보드를 관리하는 것을 돕습니다.",
    # 에이전트가 호출할 수 있는 도구 목록: 목록 조회용, 완료 처리용
    tools=[show_todos, complete_task],
)

In [16]:
# 에이전트에 입력 메시지를 전달하고 비동기로 실행한 뒤 결과를 받는다.
result = await board_agent.arun(input="지금 보드에 무엇이 올라와 있고, 각각의 상태는 어떤가요?")

# 실행 결과에서 모델이 생성한 텍스트 내용을 출력한다.
print(result.content)

현재 보드에는 항목이 하나 있습니다.

- ID 1 (목표): "notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 작성하라."  
  상태: pending  
  결과 요약: 없음 (빈값)  
  부모: 없음 (최상위 목표)

원하시면 이 작업을 시작하거나(번역 실행), 세부 단계를 추가하거나, 완료 처리할 수 있습니다. 어떤 작업을 원하시나요?


## Step 4: Add MCP

MCP is just more tools: ones you did not write, connected over a small protocol. We give the agent the filesystem reference server, the same Node server every framework uses this week, scoped to a single `workspace` folder so it can only touch files in there. In Agno an MCP server is an `MCPTools`, an async context manager you open with `async with` and pass to the agent in the same `tools=[...]` list. We build it from `StdioServerParameters` so we can set the server's working directory (`cwd`) to the workspace, which makes the agent's file names resolve there.

Agno does not expose the server's stderr, so the first line of the next cell points its stdio client at the null device. That quiets the server's startup banner and lets it run from a Jupyter kernel on Windows, where the kernel's stderr has no real file descriptor. On Mac and Linux it simply keeps the output clean.

In [18]:
# Agno는 MCP 서버의 stderr를 외부로 노출하지 않으므로, stdio 클라이언트의 errlog를 DEVNULL로 지정한다.
# 파일시스템 서버의 시작 배너 출력이 사라지고, Windows의 Jupyter 커널에서도 실행이 가능해진다.
import agno.tools.mcp.mcp as agno_mcp

# functools.partial로 stdio_client 호출 시 errlog=DEVNULL 인자를 기본값으로 고정한다.
agno_mcp.stdio_client = functools.partial(agno_mcp.stdio_client, errlog=subprocess.DEVNULL)

# 에이전트가 접근할 수 있는 유일한 폴더. 상대 경로 "workspace"를 절대 경로로 변환한다.
workspace = Path("workspace").resolve()

# 파일시스템 MCP 서버를 stdio 방식으로 실행하기 위한 파라미터를 구성한다.
server = StdioServerParameters(
    # npx로 서버 패키지를 실행한다.
    command="npx",
    # -y는 설치 확인 프롬프트를 건너뛴다. 마지막 인자로 접근 허용 폴더 하나만 전달한다.
    args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
    # 서버 프로세스의 작업 디렉터리를 workspace로 지정한다.
    cwd=str(workspace),
)

In [19]:
# 파일시스템 MCP 서버에 연결한다. 컨텍스트를 벗어나면 연결이 자동으로 종료된다.
# timeout_seconds=60은 서버 응답 대기 제한 시간을 60초로 설정한다.
async with MCPTools(server_params=server, timeout_seconds=60) as filesystem:
    # 모델, 시스템 지시문, 파일시스템 도구를 묶어 파일 작업용 에이전트를 생성한다.
    file_agent = Agent(
        model=model,
        # 에이전트의 역할과 사용 가능한 범위를 정의하는 시스템 지시문 (한글)
        instructions="당신은 자신의 workspace 안에서 파일을 읽고 쓸 수 있습니다. 도구를 사용해 요청받은 작업을 수행하십시오.",
        # 에이전트가 호출할 수 있는 도구: MCP 파일시스템 서버가 제공하는 도구 모음
        tools=[filesystem],
    )
    # 에이전트에 입력 메시지를 전달하고 비동기로 실행한 뒤 결과를 받는다.
    result = await file_agent.arun(input="notes.txt를 읽고 짧은 한 문장으로 요약해줘.")

# 컨텍스트 블록을 빠져나온 뒤, 실행 결과의 텍스트 내용을 출력한다.
print(result.content)

팀원들이 공동 보드에서 각자 과제를 골라 신중히 수행하며 작은 언어 튜터를 차근차근 만들어간다.


## Step 5: Put it in a loop with a goal

Now the payoff. Give one agent all three board tools and the filesystem server, hand it the goal, and let it run. It plans its own steps on the board, works them with its file tools, ticks each one off, and closes the goal when the work is done. That is the agent loop running on its own: read, plan, act, check off, repeat. Watch the board fill with steps and then strike them through.

In [20]:
# 에이전트에게 전달되는 시스템 지시문 (한글)
INSTRUCTIONS = """
당신은 공유 todo 보드와 파일 도구 모음을 사용하는 신중한 작업자입니다.

대기 중인(pending) 목표를 맡아 끝까지 완수하십시오. 먼저 짧은 계획을 세우는 것으로 시작합니다:
해당 작업이 실제로 나뉘는 몇 개의 구체적인 단계를 정리하여, 그 목표 아래 보드에 추가하십시오.
그런 다음 파일 도구를 사용해 각 단계를 수행하고, 하나를 끝낼 때마다 그 단계를 완료로 표시하십시오.
모든 단계가 완료되면 목표를 닫으십시오. 당신의 파일은 도구가 사용 허가를 받은 단 하나의 폴더 안에 있습니다.
"""

# 보드의 기존 내용을 모두 비우고 초기 상태로 되돌린다.
board.reset_board()

# 보드에 목표를 하나 추가하고 그 id를 받는다.
goal_id = board.add_goal("notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 작성하라.")

# 워커가 이 목표를 가져간다: 상태가 pending -> in_progress 로 바뀐다.
board.claim_todo(goal_id)

# 파일시스템 MCP 서버에 연결한다. 컨텍스트를 벗어나면 연결이 자동으로 종료된다.
async with MCPTools(server_params=server, timeout_seconds=60) as filesystem:
    # 모델, 지시문, 도구 목록(보드 도구 3개 + 파일시스템 MCP 도구)을 묶어 워커 에이전트를 생성한다.
    worker = Agent(
        model=model,
        instructions=INSTRUCTIONS,
        # 보드 조회, 단계 계획, 완료 처리, 파일 입출력 도구를 모두 제공한다.
        tools=[show_todos, plan_steps, complete_task, filesystem],
    )
    # 에이전트에 입력 메시지를 전달하고 비동기로 실행한다. 반환값은 사용하지 않는다.
    await worker.arun(input="보드에 있는 대기 중인 목표를 처리해 주세요.")

# 실행이 끝난 뒤 보드의 최종 상태를 출력한다.
board.show_board()

Goal #1: notes.txt를 읽고, 그 내용을 자연스러운 스페인어로 번역한 뒤, 그 스페인어를 spanish.txt에 작성하라.
  Step #2: notes.txt 읽기: 파일을 열어 내용을 확인한다.  notes.txt 파일을 성공적으로 읽음(초기 내용 확인).
  Step #3: 스페인어로 번역: 내용을 자연스럽고 문법적으로 올바른 스페인어로 번역한다.  notes.txt의 내용을 자연스럽고 문법적으로 올바른 스페인어로 번역하여 spanish.txt로 저장함.
  Step #4: spanish.txt에 작성: 번역 결과를 spanish.txt에 저장한다.  spanish.txt 파일 생성 및 번역 내용 확인 완료.
  Step #5: 목표 완료: 모든 단계를 완료하고 목표를 닫는다.  모든 단계 완료. 목표를 닫음.

## Run the same worker from the terminal

Everything you just watched in step 5 is also packaged as a small script, `agno_worker.py`, sitting next to this notebook. It seeds the same goal, builds the same agent with the same three board tools and the filesystem MCP server, and runs the same loop, only from the command line instead of the kernel. Open a terminal in this folder and run it:

```bash
uv run agno_worker.py
```

You will see the agent plan its steps, work the goal and tick each one off, then the finished board and the Spanish it wrote. This is the shape every worker takes on Day 5, where a Google ADK orchestrator launches one of these per framework as a parallel subprocess against a single shared board.

## The one cool thing: how fast it starts

Agno's signature claim is speed. Creating an agent is close to free, which matters when a server spins up a fresh agent per request or a team spawns many at once. Let's just measure it: build a batch of agents and time it.

```
(your numbers will vary, but each agent lands in the low microseconds)
```

In [ ]:
import time

start = time.perf_counter()
agents = [Agent(model=model, tools=[show_todos]) for _ in range(1000)]
elapsed = time.perf_counter() - start
print(f"Built {len(agents)} agents in {elapsed * 1000:.1f} ms")
print(f"That is about {elapsed / len(agents) * 1_000_000:.1f} microseconds each.")

The other half of the story is **AgentOS**: a handful of lines wraps your agents in a FastAPI app with sessions, streaming, tracing and a control-plane UI, so the same agent you just built can be served in production unchanged. And the same one-line swap as the rest of the week: Agno talks to any OpenAI-compatible endpoint through `OpenAILike(id=..., base_url=..., api_key=...)`, with the tools, the MCP server and the board all unchanged.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Seed a different goal on the board, for example "write a short haiku about Madrid into madrid.txt", and run the worker again. Does it plan sensible steps and pick the right file tools? Then run the agent with <code>await agent.aprint_response(input=..., stream=True)</code> instead of <code>arun</code>, and watch Agno print the tool-calling loop live as it happens.</span>
        </td>
    </tr>
</table>